# Benchmark: new vs. old vector-classification pipeline

Scores the **new** `rastervec` Vector-Classification + OCR pipeline against the
**old** archive/legacy pipeline on the same ground truth, using the independent
metric suite in `Evaluation/Evaluate/metrics.py` (see `EVAL_METRICS.md` for every
metric's formula). Each metric is a `Ratio(numerator, denominator)`: per-page
reports show absolute counts, aggregates are **micro-averaged** (sum numerators /
sum denominators, not the mean of per-page ratios).

- **Current (new)**: `rastervec`'s own `Pipeline.STAGES` chain (layer/color
  separation -> 12-step Vector_Classification -> FAST text detect -> PaddleOCR),
  via `pipeline.run_page_context`.
- **Archive (legacy / old)**: `archive/raster_parser/main_pipeline_extract.extract`,
  run completely unmodified via `Evaluation/Evaluate/legacy_adapter.py`.

Each `(pdf, page)` is a `PageTask` run by `rastervec.Reader.Parallel`.
`BENCH_WORKERS > 1` fans the pages across a spawn process pool (model caches
warmed once up front, so the first run is safe too).

### Two runs per page — auto vs. manual

Each pipeline (current **and** legacy) is run **twice** per page on disjoint
inputs, so the two ground-truth sources are scored against physically separate
runs and never contaminate each other:

- **auto run** — input = `convert_page_text_only` (the native text redrawn as
  vector paths, **every drawing removed**). Scored vs the `source="auto"`
  labels (`auto_label_pdf`, from the PDF's own native text).
- **manual run** — input = `convert_page_drawings_only` (the **original drawing
  vectors only**, native text removed — geometry byte-for-byte). Scored vs the
  `source="manual"` labels (`manual_label.py` sidecar `.json`). Only fires when
  the page actually has manual labels.

So `current/auto`, `current/manual`, `legacy/auto`, `legacy/manual` each get
their own aggregate and chart series.

### Dataset

`collect_dataset(DATASET_ROOT)` recursively walks one directory tree for both
`.pdf` files and label-sidecar `.json` files and pairs them into a flat
`(pdf, page)` dataset, keyed by real filesystem path — a label file that names a
root-folder PDF is the *same* item as its discovered copy, so no page runs
twice. A tree may mix labelled and unlabelled PDFs freely.

### Output

- **Per-page** evaluation reports and per-page stage timing → `RESULTS_TXT`
  (current) / `<stem>_legacy.txt` (legacy). Not printed.
- **Printed**: only the aggregated accuracy metrics, the aggregated per-stage +
  per-page timing distribution (min / Q1 / median / mean / Q3 / max), and the
  charts.
- **Per-page folder** `RECONSTRUCT_DIR/<stem>_p<N>/` (when `RECONSTRUCT_DIR` is
  set), five files: `input_auto.pdf`, `input_manual.pdf` (the two disjoint
  pipeline inputs), `current.pdf`, `legacy.pdf` (each a text reconstruction of
  that pipeline's two runs merged), and `boxes.pdf` — the current pipeline's
  pred-vs-GT overlay: **dashed** = auto GT, **solid** = manual GT, **dotted** =
  a prediction; **green** = matched, **red** = a GT no prediction reached,
  **yellow** = a prediction over no GT.

This is a **sanity/regression check**, not a real A/B: the new pipeline is
expected to score at least as well as the old one.


In [ ]:
import io
import math
import random
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "benchmark_vector_classification.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from rastervec.Evaluation.Evaluate.benchmark import (
    aggregate_results,
    distribution_stats,
    format_aggregate,
    format_aggregate_comparison,
    format_timing_report,
    format_variant_timing_comparison,
    summarize_stage_timings,
)
from rastervec.Evaluation.Evaluate.metrics import LOWER_IS_BETTER, METRIC_GROUPS, MetricConfig
from rastervec.Evaluation.Evaluate.variants import DEFAULT_VARIANTS, VARIANTS
from rastervec.logging_setup import configure_logging
from rastervec.pipeline import Pipeline
from rastervec.Reader.dataset import collect_dataset
from rastervec.Reader.Parallel import PageTask, default_worker_count, run_benchmark

configure_logging()

## Parameters

`DATASET_ROOT` is a directory tree searched **recursively** for both `.pdf`
files and label-sidecar `.json` files (the `manual_label.py` / `auto_label.py`
`LabelSet` format). `collect_dataset` pairs them into one flat `(pdf, page)`
dataset (keyed by real path, so a label file that names a root-folder PDF is the
same item as its discovered copy -- one entry per page).

`VARIANTS_TO_RUN` picks which **pipeline variants** to benchmark and compare --
names from `rastervec.Evaluation.Evaluate.variants.VARIANTS`:

- `current_heavy` / `current_light` -- the current `rastervec` pipeline with the
  full PP-OCRv6 detect+rec+orientation backend vs. the lighter
  ink-projection-segmentation + recognition-only backend.
- `current_heavy_nofast` / `current_light_nofast` -- same, but with the
  `fast_text_detect` stage turned into a pass-through (every text candidate
  reaches OCR). Slow; use `BENCH_WORKERS`.
- `legacy` -- the archive/`raster_parser` pipeline, unchanged.

Add an ablation = one entry in `variants.VARIANTS`, then list its name here.

Every page gets an **auto run** (input = native text as vectors) and, when it
has manual labels, a **manual run** (input = original drawings only), for every
selected variant -- `split_labelset_by_source` scores each run against its own
GT source.

- `RESULTS_TXT` -- per-variant per-page reports go to
  `RESULTS_TXT.with_name(stem + f"_{variant}.txt")`. Only the aggregates print.
- `RECONSTRUCT_DIR` -- per-page output folders `<stem>_p<N>_<variant>/` (`None`
  to skip). Five PDFs each.
- `BENCH_WORKERS` -- process-pool size for the per-page loop (1 = serial).

Both pipelines run real PaddleOCR, and the archive pipeline also shells out to
LibreOffice, so keep `PAGES_PER_PDF` small and `VARIANTS_TO_RUN` short.

In [ ]:
# Directory tree searched recursively for .pdf + label .json files.
DATASET_ROOT = PROJECT_ROOT / "references"

# Page cap for PDFs in the tree that have no sidecar label file (auto labels
# only). PDFs a label file references use exactly the pages it names.
PAGES_PER_PDF = 3

# Which pipeline variants to run and compare -- names from
# rastervec.Evaluation.Evaluate.variants.VARIANTS. Add an ablation there
# (one dict entry) and list its name here.
VARIANTS_TO_RUN = list(DEFAULT_VARIANTS)  # e.g. ["current_heavy", "current_light", "legacy"]

# Per-variant per-page reports -> RESULTS_TXT.with_name(stem + f"_{variant}.txt");
# the cross-variant comparison tables -> RESULTS_TXT.with_name(stem + "_comparison.txt").
# Only aggregates print.
RESULTS_TXT = PROJECT_ROOT / "benchmark_results.txt"

# Per-page output folder root (None disables). Each page x variant gets its own
# subfolder RECONSTRUCT_DIR/<stem>_p<N>_<variant>/ with five PDFs: input_auto.pdf
# / input_manual.pdf (the two disjoint pipeline inputs), current.pdf / legacy.pdf
# (text reconstructions, both runs merged), and boxes.pdf (pred-vs-GT overlay:
# dashed = auto GT, solid = manual GT, dotted = prediction; green matched,
# red = GT no prediction reached, yellow = prediction over no GT).
RECONSTRUCT_DIR = PROJECT_ROOT / "benchmark_reconstructions"

# Process-pool size for the per-page benchmark loop (1 = serial). Each worker
# loads its own PaddleOCR + torch model, so memory is the limit -- see
# default_worker_count(). run_benchmark() warms the model caches in-process
# before spawning, so BENCH_WORKERS > 1 is safe on the first run too.
BENCH_WORKERS = 1

# How many PaddleOCR cluster renders each page returns for the showcase grid
# (~50/50 OCR passes vs blank failures), and the cap on how many to plot.
SHOWCASE_PER_PAGE = 4
SHOWCASE_N = 20
SHOWCASE_SEED = 0

# MetricConfig.iou_edge_min -- minimum IoU for a gt<->prediction localisation
# edge (the N:1 fallback assignment + miss-attribution group match).
IOU_EDGE_MIN = MetricConfig().iou_edge_min

# Archive's raster-fallback stage shells out to LibreOffice (`soffice`); set
# True only if LibreOffice is installed and on PATH (only affects `legacy`).
ENABLE_ARCHIVE_RASTER_PASS = False

for _name in VARIANTS_TO_RUN:
    assert _name in VARIANTS, f"unknown variant {_name!r}; pick from {sorted(VARIANTS)}"

In [ ]:
# Recursively collect every PDF + every label file under DATASET_ROOT into one
# flat (pdf, page) dataset (see rastervec.Reader.dataset.collect_dataset).
dataset = collect_dataset(DATASET_ROOT, pages_per_pdf=PAGES_PER_PDF)

pdf_pages = [(d.pdf_path, d.page_index) for d in dataset]
# manual_entries[(pdf_path, page_index)] -> list[LabelEntry] (source == "manual")
manual_entries = {
    (d.pdf_path, d.page_index): list(d.manual_entries) for d in dataset
}

n_manual = sum(len(v) for v in manual_entries.values())
n_labelled_pages = sum(1 for v in manual_entries.values() if v)
print(f"{len(pdf_pages)} (pdf, page) pair(s) under {DATASET_ROOT}; "
      f"{n_manual} manual label(s) across {n_labelled_pages} page(s)")

In [ ]:
def build_tasks(variant: str) -> list[PageTask]:
    """One PageTask per (pdf, page) for one pipeline variant (a name from
    rastervec.Evaluation.Evaluate.variants.VARIANTS). Each task does its own
    ground truth (auto + any manual), conversion, pipeline run, evaluation
    and per-page PDF output -- see rastervec.Reader.Parallel.benchmark_jobs."""
    is_current = VARIANTS[variant].engine == "current"
    return [
        PageTask(
            pdf_path=pdf_path,
            page_index=page_index,
            manual_entries=list(manual_entries.get((pdf_path, page_index), [])),
            iou_edge_min=IOU_EDGE_MIN,
            variant=variant,
            reconstruct_dir=str(RECONSTRUCT_DIR) if RECONSTRUCT_DIR else None,
            showcase_per_page=SHOWCASE_PER_PAGE if is_current else 0,
            enable_archive_raster_pass=ENABLE_ARCHIVE_RASTER_PASS,
            showcase_seed=SHOWCASE_SEED,
        )
        for pdf_path, page_index in pdf_pages
    ]


def collect_results(results: list, txt_path: Path) -> tuple[list, list, list, list]:
    """Split one variant's PageResults into (auto, manual, stage_timings,
    showcase) and write every per-page report block + failure to txt_path.
    A variant with no per-stage breakdown (legacy) contributes a
    {"pipeline_total": total_seconds} entry so its total still compares."""
    auto, manual, timings, showcase, lines = [], [], [], [], []
    for r in results:
        if r.error is not None:
            lines.append(f"[{r.variant}] {r.pdf_path} page {r.page_index} failed: {r.error}")
            continue
        lines.extend(r.report_blocks)
        lines.append(
            f"  stage timing: total={r.total_seconds:.2f}s"
            + (f"  ({', '.join(f'{k}={v:.2f}s' for k, v in r.stage_durations.items())})"
               if r.stage_durations else "")
        )
        if r.auto is not None:
            auto.append(r.auto)
        if r.manual is not None:
            manual.append(r.manual)
        timings.append(r.stage_durations or {"pipeline_total": r.total_seconds})
        showcase.extend(r.showcase)
    txt_path.write_text("\n\n".join(lines) + "\n", encoding="utf-8")
    return auto, manual, timings, showcase

## Run the selected variants

Each variant in `VARIANTS_TO_RUN` is benchmarked over the whole dataset (auto +
any manual run per page). Results land in `*_by_variant` dicts keyed by variant
name; the comparison cells below read those.

In [ ]:
results_by_variant, auto_by_variant, manual_by_variant = {}, {}, {}
timings_by_variant, showcase_by_variant = {}, {}

for _v in VARIANTS_TO_RUN:
    _txt = RESULTS_TXT.with_name(RESULTS_TXT.stem + f"_{_v}.txt")
    _res = run_benchmark(build_tasks(_v), workers=BENCH_WORKERS, desc=_v)
    _auto, _manual, _timings, _showcase = collect_results(_res, _txt)
    results_by_variant[_v] = _res
    auto_by_variant[_v] = _auto
    manual_by_variant[_v] = _manual
    timings_by_variant[_v] = _timings
    showcase_by_variant[_v] = _showcase
    print(f"[{_v}] {len(_auto)} auto, {len(_manual)} manual page-score(s); "
          f"{sum(1 for r in _res if r.error)} failed  ->  {_txt.name}")

### PaddleOCR render showcase

Up to `SHOWCASE_PER_PAGE` cluster renders per page (sampled ~50/50 between
non-blank and blank OCR readings), from the first `current_*` variant that
produced a showcase pool.

In [ ]:
_current_variants = [v for v in VARIANTS_TO_RUN if VARIANTS[v].engine == "current"]
_showcase_variant = next((v for v in _current_variants if showcase_by_variant.get(v)), None)
pool = showcase_by_variant.get(_showcase_variant, []) if _showcase_variant else []
passed = [s for s in pool if s.passed]
failed = [s for s in pool if not s.passed]
print(f"showcase variant: {_showcase_variant}  |  pool: {len(pool)} renders "
      f"({len(passed)} passed, {len(failed)} blank)")

rng = random.Random(SHOWCASE_SEED)
half = SHOWCASE_N // 2
pick = rng.sample(passed, min(half, len(passed)))
pick += rng.sample(failed, min(SHOWCASE_N - len(pick), len(failed)))
chosen = {id(s) for s in pick}
rest = [s for s in pool if id(s) not in chosen]
rng.shuffle(rest)
pick += rest[: max(0, SHOWCASE_N - len(pick))]
rng.shuffle(pick)

if not pick:
    print("nothing to showcase -- run the benchmark cell first")
else:
    cols = 4
    rows = (len(pick) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 2.4 * rows))
    axes = axes.ravel() if hasattr(axes, "ravel") else [axes]
    for ax in axes:
        ax.axis("off")
    for ax, s in zip(axes, pick):
        ax.imshow(Image.open(io.BytesIO(s.png)))
        ax.set_title((s.text or "(blank)")[:40], fontsize=8,
                     color="#2a7" if s.passed else "#c33")
    plt.tight_layout()
    plt.show()

## Compare variants

Aggregate metrics (micro-averaged) and per-stage median wall-clock, side by side
across `VARIANTS_TO_RUN`. The timing table's `d:<variant>` columns are the delta
vs. the first variant in the list.

In [ ]:
_auto_aggs = {v: aggregate_results(auto_by_variant.get(v, [])) for v in VARIANTS_TO_RUN}
print(format_aggregate_comparison(_auto_aggs, title="Aggregate metrics by variant -- AUTO ground truth"))
print()

if any(manual_by_variant.get(v) for v in VARIANTS_TO_RUN):
    _manual_aggs = {v: aggregate_results(manual_by_variant.get(v, [])) for v in VARIANTS_TO_RUN}
    print(format_aggregate_comparison(_manual_aggs, title="Aggregate metrics by variant -- MANUAL ground truth"))
    print()

timing_summaries = {
    v: summarize_stage_timings(timings_by_variant.get(v, []), Pipeline.stage_keys())
    for v in VARIANTS_TO_RUN
}
print(format_variant_timing_comparison(timing_summaries))

In [ ]:
# Write the cross-variant comparison tables to their own .txt.
_comparison_txt = RESULTS_TXT.with_name(RESULTS_TXT.stem + "_comparison.txt")
_blocks = [
    format_aggregate_comparison(
        {v: aggregate_results(auto_by_variant.get(v, [])) for v in VARIANTS_TO_RUN},
        title="Aggregate metrics by variant -- AUTO ground truth",
    ),
    format_variant_timing_comparison(timing_summaries),
]
_comparison_txt.write_text("\n\n".join(_blocks) + "\n", encoding="utf-8")
print(f"comparison tables -> {_comparison_txt}")

## Metric chart -- one series per variant (auto ground truth)

In [ ]:
import math

import matplotlib.pyplot as plt

_series = [
    (v, aggregate_results(auto_by_variant.get(v, [])))
    for v in VARIANTS_TO_RUN
]
_series = [(name, agg) for name, agg in _series if agg is not None]

_DOWN = " " + chr(0x2193)  # down arrow = "lower is better"
_groups = list(METRIC_GROUPS)
fig, axes = plt.subplots(len(_groups), 1, figsize=(11, 3.1 * len(_groups)))
if len(_groups) == 1:
    axes = [axes]

for ax, (dimension, names) in zip(axes, _groups):
    x = range(len(names))
    n = max(len(_series), 1)
    width = 0.8 / n
    for i, (sname, agg) in enumerate(_series):
        offset = (i - (n - 1) / 2) * width
        vals = []
        for m in names:
            v = agg.get(m)
            vals.append(0.0 if (v is None or math.isnan(v)) else v)
        ax.bar([j + offset for j in x], vals, width, label=sname)
    ax.set_xticks(list(x))
    ax.set_xticklabels(
        [m + (_DOWN if m in LOWER_IS_BETTER else "") for m in names],
        rotation=20, ha="right", fontsize=7,
    )
    ax.set_ylim(0, 1)
    ax.set_title(dimension)
    ax.legend(fontsize=7, loc="lower right")

fig.suptitle("Metric suite by variant (auto GT; down arrow = lower is better)")
plt.tight_layout()
plt.show()

## Per-stage timing chart -- grouped bars by variant

In [ ]:
import matplotlib.pyplot as plt

_summaries = {v: s for v, s in timing_summaries.items() if s}
_stages = list(Pipeline.stage_keys())

if not _summaries:
    print("no timing data -- run the benchmark cell first")
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    n = len(_summaries)
    width = 0.8 / max(n, 1)
    for i, (v, s) in enumerate(_summaries.items()):
        offset = (i - (n - 1) / 2) * width
        med = [s.get(k, {}).get("median", 0.0) for k in _stages]
        ax.bar([j + offset for j in range(len(_stages))], med, width, label=v)
    ax.set_xticks(range(len(_stages)))
    ax.set_xticklabels(_stages, rotation=30, ha="right")
    ax.set_ylabel("seconds (median per page)")
    ax.legend(fontsize=8)
    ax.set_title("Per-stage median wall-clock by variant")
    plt.tight_layout()
    plt.show()

## Reading the results

- `current_light` is the default pipeline OCR path (ink-projection line/word
  segmentation + PaddleOCR recognition-only + 48px normalization);
  `current_heavy` is the old full PP-OCRv6 detect+rec+orientation backend.
  `*_nofast` skips `fast_text_detect` (every candidate reaches OCR).
- `legacy` is the archive pipeline, unchanged. It has no per-stage timing
  breakdown -- its timing column shows `nan` for stages and only the `total`
  row is comparable.
- **auto vs. manual**: each is scored from a separate pipeline run on its own
  input (text-only vs drawings-only), so predictions never cross sources.
- **`RECONSTRUCT_DIR/<stem>_p<N>_<variant>/boxes.pdf`**: dashed = auto GT,
  solid = manual GT, dotted = a prediction; green = matched, red = a GT no
  prediction reached, yellow = a prediction over no GT.